# NumCompute Stream Demo

This notebook demonstrates the NumCompute Stream package.

It shows:

- Loading a CSV dataset using custom `io.py`
- Splitting data into train/test sets
- Splitting data into chunks for streaming learning
- Training a single decision tree with `.partial_fit()`
- Training an ensemble model with `.partial_fit()`
- Logging accuracy over chunks
- Visualising metrics using `visualise.py`
- Showing final predictions and test accuracy

In [ ]:
import sys
from pathlib import Path

import numpy as np

current_dir = Path.cwd()

if current_dir.name == "demo":
    project_root = current_dir.parent
    demo_dir = current_dir
else:
    project_root = current_dir
    demo_dir = project_root / "demo"

sys.path.append(str(project_root))

from numcompute_stream.io import load_csv, train_test_split, make_chunks
from numcompute_stream.preprocessing import SimpleImputer, StandardScaler
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.metrics import Accuracy
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
    plot_error_over_time,
)

print("Imports successful")
print("Project root:", project_root)
print("Demo folder:", demo_dir)

In [ ]:
dataset_path = demo_dir / "sample_data.csv"

X, y = load_csv(
    file_path=str(dataset_path),
    target_column=-1,
    has_header=True,
)

print("Dataset loaded successfully using custom io.py")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst five rows of X:")
print(X[:5])

print("\nFirst five labels:")
print(y[:5])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    shuffle=True,
    random_state=42,
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

In [ ]:
chunk_size = 6

train_chunks = list(make_chunks(
    X_train,
    y_train,
    chunk_size=chunk_size,
))

print("Number of streaming chunks:", len(train_chunks))

for i, (X_chunk, y_chunk) in enumerate(train_chunks, start=1):
    print(f"Chunk {i}: X={X_chunk.shape}, y={y_chunk.shape}")

In [ ]:
tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(
        max_depth=4,
        min_samples_split=2,
        criterion="gini",
        random_state=42,
    )),
])

print("Single decision tree pipeline created")

In [ ]:
ensemble_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", EnsembleClassifier(
        n_estimators=5,
        max_depth=4,
        min_samples_split=2,
        criterion="gini",
        max_features="sqrt",
        bootstrap=True,
        random_state=42,
    )),
])

print("Ensemble pipeline created")

In [ ]:
tree_trainer = StreamTrainer(
    pipeline=tree_pipeline,
    metrics={"accuracy": Accuracy()},
    name="Single Decision Tree",
)

ensemble_trainer = StreamTrainer(
    pipeline=ensemble_pipeline,
    metrics={"accuracy": Accuracy()},
    name="Bagging Ensemble",
)

print("Stream trainers created")

In [ ]:
for chunk_number, (X_chunk, y_chunk) in enumerate(train_chunks, start=1):
    print(f"\nProcessing chunk {chunk_number}")

    tree_log = tree_trainer.fit_score_chunk(X_chunk, y_chunk)
    ensemble_log = ensemble_trainer.fit_score_chunk(X_chunk, y_chunk)

    print("Tree log:", tree_log)
    print("Ensemble log:", ensemble_log)

In [ ]:
print("Tree summary:")
print(tree_trainer.summary())

print("\nEnsemble summary:")
print(ensemble_trainer.summary())

In [ ]:
tree_accuracy = tree_trainer.get_metric_history("accuracy")
ensemble_accuracy = ensemble_trainer.get_metric_history("accuracy")

print("Tree accuracy history:")
print(tree_accuracy)

print("\nEnsemble accuracy history:")
print(ensemble_accuracy)

In [ ]:
plot_metric_over_time(
    tree_accuracy,
    title="Single Decision Tree Accuracy Over Streaming Chunks",
    ylabel="Accuracy",
)

In [ ]:
compare_models(
    tree_accuracy,
    ensemble_accuracy,
    labels=("Single Tree", "Bagging Ensemble"),
    title="Single Tree vs Ensemble Accuracy",
    ylabel="Accuracy",
)

In [ ]:
plot_error_over_time(
    ensemble_accuracy,
    title="Ensemble Error Over Streaming Chunks",
)

In [ ]:
tree_predictions = tree_pipeline.predict(X_test)
ensemble_predictions = ensemble_pipeline.predict(X_test)

print("True labels:")
print(y_test)

print("\nTree predictions:")
print(tree_predictions)

print("\nEnsemble predictions:")
print(ensemble_predictions)

In [ ]:
tree_test_accuracy = np.mean(tree_predictions == y_test)
ensemble_test_accuracy = np.mean(ensemble_predictions == y_test)

print("Final test accuracy")
print("Single decision tree:", tree_test_accuracy)
print("Bagging ensemble:", ensemble_test_accuracy)

In [ ]:
plot_predictions_vs_ground_truth(
    y_test,
    ensemble_predictions,
    title="Ensemble Predictions vs Ground Truth",
)

## Conclusion

This demo shows that the NumCompute Stream package can load CSV data using a custom I/O module, split data into chunks, train models incrementally using `.partial_fit()`, log streaming accuracy, compare a single decision tree with an ensemble model, and visualise results over time using matplotlib.

This satisfies the assignment requirements for streaming learning, model ensembling, pipeline usage, metrics, and built-in visualisation.